In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from fredapi import Fred
import yfinance as yf

## Data Collection

Data for this project will be obtained from FRED (Federal Reserve Economic Data), Yahoo Finance APIs, and historical CSV files for indicators where API coverage does not extend back to 1960. The following macroeconomic and financial indicators will be examined:

### FRED Indicators
- **Unemployment Rate (UNRATE)** - monthly measure of the percentage of the labour force that is unemployed, a key lagging indicator of economic health
- **Consumer Price Index (CPIAUCSL)** - measures inflation through changes in the price of a basket of consumer goods and services
- **Federal Funds Rate (FEDFUNDS)** - the interest rate at which banks lend to each other overnight, a primary tool of US monetary policy
- **Industrial Production (INDPRO)** - measures real output of the manufacturing, mining and utilities sectors, a leading indicator of economic activity
- **House Prices (CSUSHPISA)** - S&P CoreLogic Case-Shiller US National Home Price Index, captures housing market conditions; available from 1987
- **Yield Curve (T10Y2Y)** - spread between 10-year and 2-year Treasury yields; inversion has preceded every US recession since the 1950s; available from 1976
- **Initial Jobless Claims (ICSA)** - weekly count of new unemployment insurance claims, one of the most timely labour market indicators available; available from 1967
- **Credit Spread (BAA10Y)** - spread between Moody's BAA corporate bond yield and 10-year Treasury yield; widens as default risk rises ahead of recessions; available from 1986
- **Consumer Credit (TOTALSL)** - total outstanding consumer credit, captures household borrowing trends
- **Housing Starts (HOUST)** - number of new residential construction projects begun each month, a forward-looking indicator of economic activity
- **30-Year Fixed Mortgage Rate (MORTGAGE30US)** - captures consumer borrowing costs in the housing market; available from 1971
- **Money Supply M2 (M2SL)** - broad measure of money supply including cash, deposits and money market funds; changes in M2 often precede turning points in economic activity
- **Federal Debt as % of GDP (GFDEGDQ188S)** - captures long-term fiscal sustainability; quarterly series interpolated to monthly frequency; available from 1966
- **Corporate Profits (CP)** - pre-tax corporate profits, a leading indicator of business investment and hiring decisions; quarterly series interpolated to monthly frequency
- **Recession Indicator (USREC)** - NBER defined binary indicator, 1 during recession months and 0 otherwise; used as the target variable

### Yahoo Finance and CSV Indicators
- **S&P 500 (^GSPC)** - broad measure of US equity market performance, reflects investor expectations about future economic conditions; extended back to 1960 using historical CSV data combined with Yahoo Finance API
- **Gold** - commodity price commonly associated with safe haven demand during periods of economic uncertainty; extended back to 1960 using historical CSV data combined with Yahoo Finance API


In [2]:
# FRED API Connection

FRED_API_KEY = "99a4258f3c059800db6f3473af4d2edf"
fred = Fred(api_key=FRED_API_KEY)

In [3]:
# Start & End Dates for Project

START = "1960-01-01"
END   = pd.Timestamp.today().strftime("%Y-%m-%d")

In [4]:
# FRED Indicators 

unemployment          = fred.get_series("UNRATE",       START, END)
cpi                   = fred.get_series("CPIAUCSL",     START, END)
fed_funds             = fred.get_series("FEDFUNDS",     START, END)
industrial_production = fred.get_series("INDPRO",       START, END)
house_prices          = fred.get_series("CSUSHPISA",    START, END)
recession             = fred.get_series("USREC",        START, END)
yield_curve           = fred.get_series("T10Y2Y",       START, END)
jobless_claims        = fred.get_series("ICSA",         START, END)
credit_spread         = fred.get_series("BAA10Y",       START, END)
consumer_credit       = fred.get_series("TOTALSL",      START, END)
lei                   = fred.get_series("USSLIND",      START, END)
housing_starts        = fred.get_series("HOUST",        START, END)
mortgage_rates        = fred.get_series("MORTGAGE30US", START, END)
debt_gdp              = fred.get_series("GFDEGDQ188S",  START, END)
corporate_profits     = fred.get_series("CP",           START, END)
m2                    = fred.get_series("M2SL",         START, END)

print("FRED data collected")

FRED data collected


In [5]:
# Yahoo Finance Indicators 

sp500_raw  = yf.download("^GSPC",  start=START, end=END, interval="1mo", progress=False)
gold_raw   = yf.download("GC=F",   start=START, end=END, interval="1mo", progress=False)
vix_raw    = yf.download("^VIX",   start=START, end=END, interval="1mo", progress=False)

sp500  = sp500_raw["Close"].squeeze()
gold   = gold_raw["Close"].squeeze()
vix    = vix_raw["Close"].squeeze()

print("Yahoo Finance data collected")

Yahoo Finance data collected


In [6]:
# Align to Monthly Frequency 
# resample all series to month start, forward fill where needed

def to_monthly(series, label):
    s = series.copy()
    s.index = pd.to_datetime(s.index)
    s = s.resample("MS").mean()
    s.name = label
    return s

unemployment          = to_monthly(unemployment,          "unemployment")
cpi                   = to_monthly(cpi,                   "cpi")
fed_funds             = to_monthly(fed_funds,             "fed_funds")
industrial_production = to_monthly(industrial_production, "industrial_production")
house_prices          = to_monthly(house_prices,          "house_prices")
recession             = to_monthly(recession,             "recession")
yield_curve           = to_monthly(yield_curve,           "yield_curve")
jobless_claims        = to_monthly(jobless_claims,        "jobless_claims")
credit_spread         = to_monthly(credit_spread,         "credit_spread")
consumer_credit       = to_monthly(consumer_credit,       "consumer_credit")
lei                   = to_monthly(lei,                   "lei")
housing_starts        = to_monthly(housing_starts,        "housing_starts")
mortgage_rates        = to_monthly(mortgage_rates,        "mortgage_rates")
sp500                 = to_monthly(sp500,                 "sp500")
gold                  = to_monthly(gold,                  "gold")
vix                   = to_monthly(vix,                   "vix")
m2                    = to_monthly(m2,                    "m2")


# quarterly indicators - resample then interpolate
debt_gdp = fred.get_series("GFDEGDQ188S", START, END)
debt_gdp = pd.Series(debt_gdp.values, index=pd.to_datetime(debt_gdp.index))
debt_gdp = debt_gdp.resample("MS").interpolate(method="time")
debt_gdp.name = "debt_gdp"

corporate_profits = fred.get_series("CP", START, END)
corporate_profits = pd.Series(corporate_profits.values, index=pd.to_datetime(corporate_profits.index))
corporate_profits = corporate_profits.resample("MS").interpolate(method="time")
corporate_profits.name = "corporate_profits"

print("All series aligned to monthly frequency")

All series aligned to monthly frequency


In [7]:
# Data Coverage Summary 

series_dict = {
    "unemployment":           unemployment,
    "cpi":                    cpi,
    "fed_funds":              fed_funds,
    "industrial_production":  industrial_production,
    "house_prices":           house_prices,
    "recession":              recession,
    "yield_curve":            yield_curve,
    "jobless_claims":         jobless_claims,
    "credit_spread":          credit_spread,
    "consumer_credit":        consumer_credit,
    "lei":                    lei,
    "housing_starts":         housing_starts,
    "mortgage_rates":         mortgage_rates,
    "sp500":                  sp500,
    "gold":                   gold,
    "vix":                    vix,
    "debt_gdp":               debt_gdp,
    "corporate_profits":      corporate_profits,
    "m2":                     m2,
}

print(f"{'Indicator':<25} {'Start':<15} {'End':<15} {'Rows':<10} {'NaNs'}")
print("-" * 75)
for name, series in series_dict.items():
    clean = series.dropna()
    nans  = series.isnull().sum()
    print(f"{name:<25} {str(clean.index.min().date()):<15} {str(clean.index.max().date()):<15} {len(clean):<10} {nans}")

Indicator                 Start           End             Rows       NaNs
---------------------------------------------------------------------------
unemployment              1960-01-01      2026-05-01      796        1
cpi                       1960-01-01      2026-05-01      796        1
fed_funds                 1960-01-01      2026-06-01      798        0
industrial_production     1960-01-01      2026-05-01      797        0
house_prices              1987-01-01      2026-03-01      471        144
recession                 1960-01-01      2026-06-01      798        0
yield_curve               1976-06-01      2026-07-01      602        0
jobless_claims            1967-01-01      2026-06-01      714        0
credit_spread             1986-01-01      2026-06-01      486        0
consumer_credit           1960-01-01      2026-04-01      796        0
lei                       1982-01-01      2020-02-01      458        0
housing_starts            1960-01-01      2026-05-01      797      

## Data Availability and Limitations

Many of the indicators used in this project were not historically recorded from 1960, which presents a key challenge in constructing a consistent long-run dataset. The table below summarises the earliest available data for each indicator based on the coverage summary output:

| Indicator | Source | Available From | Notes |
|---|---|---|---|
| Unemployment | FRED | 1960 | |
| CPI | FRED | 1960 | |
| Federal Funds Rate | FRED | 1960 | |
| Industrial Production | FRED | 1960 | |
| Consumer Credit | FRED | 1960 | |
| Housing Starts | FRED | 1960 | |
| Corporate Profits | FRED | 1960 | Quarterly, interpolated to monthly |
| M2 Money Supply | FRED | 1960 | |
| Gold | Historical CSV + Yahoo Finance | 1960 | CSV extended back to 1960, Yahoo Finance fills forward |
| S&P 500 | Historical CSV + Yahoo Finance | 1960 | CSV extended back to 1960, Yahoo Finance fills forward |
| Jobless Claims | FRED | 1967 | |
| Mortgage Rates | FRED | 1971 | |
| Yield Curve | FRED | 1976 | |
| House Prices | FRED | 1987 | 144 NaNs before 1987, excluded from 1960 dataset |
| Debt as % of GDP | FRED | 1966 | Quarterly, interpolated to monthly |
| Credit Spread | FRED | 1986 | |

Two indicators were considered but excluded from the final datasets:

- **VIX** - only available from 1990, insufficient historical coverage for the 1960 dataset and partially redundant given credit spread and S&P 500 already capture market stress
- **LEI (USSLIND)** - OECD composite leading indicator available from 1982 but series ends in 2020, making it unsuitable for a project that includes data up to the present day

These availability constraints directly influenced the dataset structure adopted in this project. Rather than restricting all indicators to the shortest common time period, which would reduce the dataset to approximately 36 years of data and lose several important recession episodes, two datasets were constructed:

- **1960 Dataset** - contains indicators available from 1960, covering all nine NBER recession episodes from 1960 to present. This dataset prioritises historical depth over indicator breadth. Indicators with limited historical coverage such as house prices, yield curve, credit spread and mortgage rates are excluded.
- **1987 Dataset** - contains the full indicator set from 1987 onwards, covering four recession episodes: the Gulf War recession, the Dot-Com recession, the Global Financial Crisis, and COVID-19. This dataset sacrifices historical length in favour of a richer feature set reflecting the modern US economy.

Where minor gaps exist within a series, time-based interpolation was applied to preserve continuity. Quarterly indicators including corporate profits and federal debt as a percentage of GDP were resampled to monthly frequency using interpolation to create smooth monthly estimates between quarterly release points. No large-scale imputation was used for structural gaps where data genuinely did not exist.

In [8]:
# Gold and S&P 500 historical data was sourced via macrotrends

gold_hist = pd.read_csv("Gold_Price_US.csv", parse_dates=["Date"])
sp500_hist = pd.read_csv("S&P500.csv", parse_dates=["Date"])

#https://www.macrotrends.net/datasets/1333/historical-gold-prices-100-year-chart
#https://www.macrotrends.net/2324/sp-500-historical-chart-data

In [9]:
gold_hist

,Date,Value
0,1915-01-01,19.2500
1,1915-02-01,19.2500
2,1915-03-01,19.2500
3,1915-04-01,19.2500
4,1915-05-01,19.2500
...,...,...
1334,2026-03-01,4670.3486
1335,2026-04-01,4629.5648
1336,2026-05-01,4541.8773
1337,2026-06-01,4006.7000


In [10]:
sp500_hist

,Date,Value
0,1927-12-31,17.66
1,1928-01-31,17.57
2,1928-02-29,17.26
3,1928-03-31,19.28
4,1928-04-30,19.75
...,...,...
1178,2026-02-27,6878.88
1179,2026-03-31,6528.52
1180,2026-04-30,7209.01
1181,2026-05-29,7580.06


In [11]:
gold_hist = gold_hist.set_index("Date")
gold_hist.index = pd.to_datetime(gold_hist.index)
gold_hist = gold_hist.rename(columns={"Value": "gold"})
gold_hist = gold_hist.resample("MS").mean()

sp500_hist = sp500_hist.set_index("Date")
sp500_hist.index = pd.to_datetime(sp500_hist.index)
sp500_hist = sp500_hist.rename(columns={"Value": "sp500"})
sp500_hist = sp500_hist.resample("MS").mean()

In [12]:
# Gold - CSV + Yahoo Finance 

gold_csv_end = gold_hist.index.max()

gold_csv_end_check = pd.Timestamp.today().normalize()
if gold_csv_end < gold_csv_end_check - pd.DateOffset(months=1):
    gold_yahoo_raw = yf.download("GC=F", start=gold_csv_end + pd.DateOffset(months=1),
                                  end=END, interval="1mo", progress=False)
    gold_yahoo = gold_yahoo_raw["Close"].squeeze()
    gold_yahoo.name = "gold"
    gold_yahoo.index = pd.to_datetime(gold_yahoo.index)
    gold_yahoo = gold_yahoo.resample("MS").mean()
    gold = pd.concat([gold_hist["gold"], gold_yahoo])
else:
    gold = gold_hist["gold"]

gold = gold[~gold.index.duplicated(keep="first")]
gold.name = "gold"
gold = gold["1960-01-01":]

print(f"Gold: {gold.index.min().date()} to {gold.index.max().date()} - {len(gold)} rows")

# S&P 500 - CSV + Yahoo Finance 
# CSV dates are end of month - convert to month start

sp500_hist.index = sp500_hist.index + pd.offsets.MonthBegin(1)

sp500_csv_end = sp500_hist.index.max()

if sp500_csv_end < gold_csv_end_check - pd.DateOffset(months=1):
    sp500_yahoo_raw = yf.download("^GSPC", start=sp500_csv_end + pd.DateOffset(months=1),
                                   end=END, interval="1mo", progress=False)
    if not sp500_yahoo_raw.empty:
        sp500_yahoo = sp500_yahoo_raw["Close"]
        if isinstance(sp500_yahoo, pd.DataFrame):
            sp500_yahoo = sp500_yahoo.iloc[:, 0]
        sp500_yahoo = sp500_yahoo.squeeze()
        if isinstance(sp500_yahoo, pd.Series):
            sp500_yahoo.name = "sp500"
            sp500_yahoo.index = pd.to_datetime(sp500_yahoo.index)
            sp500_yahoo = sp500_yahoo.resample("MS").mean()
            sp500 = pd.concat([sp500_hist["sp500"], sp500_yahoo])
        else:
            sp500 = sp500_hist["sp500"]
    else:
        sp500 = sp500_hist["sp500"]
else:
    sp500 = sp500_hist["sp500"]

sp500 = sp500[~sp500.index.duplicated(keep="first")]
sp500.name = "sp500"
sp500 = sp500["1960-01-01":]

print(f"S&P 500: {sp500.index.min().date()} to {sp500.index.max().date()} - {len(sp500)} rows")

Gold: 1960-01-01 to 2026-07-01 - 799 rows
S&P 500: 1960-01-01 to 2026-07-01 - 799 rows
